# Session 2: Pandas, Tidy Data & Entity Matching
**Data Science Techniques and Real-World Applications &mdash; WS 2026**

**Zahra Sharafi**

**Frankfurt School of Finance and Management**

Thursday 10 September 2026 

Today's dataset: daily prices and returns for four tech stocks (2018&ndash;2024), plus a small
company-info table we'll build ourselves.


## <span style="color:#1e3a8a">Tidy data</span>

Before touching any code: what makes a dataset easy to work with?

A **tidy** dataset is organized so that:
1. Each observation forms a *row*.
2. Each variable forms a *column -- Features and label/target*.
3. Each kind of observation (individulas, products, etc.) forms its own *table*.
4. In *relational* datasets, the link between tables is clear and documented.

**Types of variables**
- **Quantitative** &mdash; born as numbers (integers, floats). *Flow* variables are measured over a
  time frame (e.g. monthly sales); *stock* variables are measured at a point in time (e.g. a closing price).
- **Qualitative / categorical** &mdash; labels, or numbers used as labels (e.g. `0 = US, 1 = EU, 2 = RoW`).
  A **binary** variable (0/1) is a special case: the numbers aren't arbitrary in the same way as categorical, and the mean is meaningful and it's already numeric-ready 

**Types of observations**

  | Observation type | Meaning | Example |
  |---|---|---|
  | Cross-sectional (xsec) | different units observed at the same time | 4 companies' closing prices on one day |
  | Time series (tseries) | one unit observed over time | AAPL's daily price, 2018&ndash;2024 |
  | Panel (xt / longitudinal) | many units observed over time | our full tech-stocks dataset: 4 tickers &times; many days |


## <span style="color:#1e3a8a">Pandas essentials</span>

Pandas is a powerful open-source Python library designed for efficient and intuitive data manipulation and analysis. It provides data structures and functions that make working with structured data simple and expressive. The name "Pandas" is derived from the term "Panel Data," which refers to multidimensional structured datasets.

Pandas is the standard library for tabular data in Python &mdash; a DataFrame is essentially a tidy
data table with row and column labels.



In [34]:
import pandas as pd
import numpy as np

### Creating a DataFrame

A **Series** is a single labeled column; a **DataFrame** is several Series sharing one index.

A Series is a NumPy 1D array plus a label (index) for each element, plus some pandas-specific extras (missing-value handling, alignment by label, non-NumPy dtypes like categoricals or timezone-aware dates).

In [8]:
observations = [10, 18, 23, 44]
my_series = pd.DataFrame({"A": observations})
my_series

,A
0,10
1,18
2,23
3,44


The index can be changed to other numbers, strings, or dates (useful for time series):

In [9]:
my_series.index = range(10, 14)
my_series

,A
10,10
11,18
12,23
13,44


A DataFrame is just several Series sharing one index -- add more columns the same way:

In [10]:
my_df = pd.DataFrame({"A": [10, 18, 23, 44], "B": [11, 13, 18, 30]})
my_df.index = ["a", "b", "c", "d"]      # relabel the index
my_df.columns = ["Aa", "Bb"]              # relabel the columns
my_df

,Aa,Bb
a,10,11
b,18,13
c,23,18
d,44,30


### Reading data

In practice you'll load data from a file far more often than you'll type it in by hand. Pandas reads most common formats with a `pd.read_*` function -- `read_csv`, `read_excel`, `read_json`, `read_sql`, and more ([full list](https://pandas.pydata.org/docs/reference/io.html)). They all return a DataFrame the same way, whatever the source format:

In [11]:
df = pd.read_excel("data/techstocks.xlsx", sheet_name="main")
df.head()

,PERMNO,Names Date,Ticker Symbol,Company Name,Price or Bid/Ask Average,Returns,Return on the S&P 500 Index
0,14542,2018-01-02,GOOG,ALPHABET INC,1065.000000,0.017775,0.008303
1,14542,2018-01-03,GOOG,ALPHABET INC,1082.479980,0.016413,0.006399
2,14542,2018-01-04,GOOG,ALPHABET INC,1086.400024,0.003621,0.004029
3,14542,2018-01-05,GOOG,ALPHABET INC,1102.229980,0.014571,0.007034
4,14542,2018-01-08,GOOG,ALPHABET INC,1106.939941,0.004273,0.001662


What are we looking at? One row per (ticker, day) &mdash; a panel, per the table above.

In [12]:
df.shape

(3024, 7)

In [13]:
df.dtypes

PERMNO                                  int64
Names Date                     datetime64[ns]
Ticker Symbol                          object
Company Name                           object
Price or Bid/Ask Average              float64
Returns                               float64
Return on the S&P 500 Index           float64
dtype: object

<span style="color:#b45309">**Exercise 1: Get to know your DataFrame**</span>

Before diving into specific tools, get comfortable with `df` itself. Using `.shape`, `.dtypes`, and `.describe()`, answer:

1. How many rows and columns does it have?
2. What are the data types of each column?
3. What are the earliest and latest dates in `Names Date`? (hint: `.min()` / `.max()` work on a date column too)
4. What are the mean and standard deviation of `Price or Bid/Ask Average`?

Try it in the cell below, then check the answer notebook (`Session 2a - Exercise Answers.ipynb`).


In [ ]:
# your code here


### Indexing & filtering

In [4]:
# a single column -> a Series; unique values in it
df["Ticker Symbol"].unique()

array(['GOOG', 'AAPL', 'AMZN', 'NFLX'], dtype=object)

`.value_counts()` goes one step further than `.unique()` -- it counts how many rows fall into each value:

In [ ]:
df["Ticker Symbol"].value_counts()

Sorting a DataFrame is just `.sort_values()`:

In [ ]:
df.sort_values("Returns", ascending=False).head()   # biggest single-day gains first

Two ways to select directly:
- **`.loc[]`** selects by **label** (index value / column name)
- **`.iloc[]`** selects by **integer position**, regardless of what the label actually is

On a freshly loaded DataFrame the two usually agree (the row in position 0 is also *labeled* 0) -- but once you filter or sort, positions and labels diverge, and the difference starts to matter. You'll see exactly that in the next cell, which is why it ends with `.reset_index(drop=True)`.

In [ ]:
df.loc[0]     # the row labeled 0

In [ ]:
df.iloc[0]    # the row in the first position -- same result here, before any filtering

<span style="color:#b45309">**Exercise 2: `.loc[0]` vs `.iloc[0]` after sorting -- same row, or not?**</span>

Sort `df` by `Returns`, descending, into `best_day`. Print `best_day.iloc[0]` (the row in the first *position*) and `best_day.loc[0]` (the row *labeled* 0). Are they the same row? Why, or why not?

Try it in the cell below, then check the answer notebook (`Session 2a - Exercise Answers.ipynb`).


In [ ]:
# your code here


In [45]:
# filter to one ticker
aapl = df[df["Ticker Symbol"] == "AAPL"].reset_index(drop=True)
aapl.head()

,PERMNO,Names Date,Ticker Symbol,Company Name,Price or Bid/Ask Average,Returns,Return on the S&P 500 Index
0,14593,2018-01-02,AAPL,APPLE INC,172.259995,0.017905,0.008303
1,14593,2018-01-03,AAPL,APPLE INC,172.229996,-0.000174,0.006399
2,14593,2018-01-04,AAPL,APPLE INC,173.029999,0.004645,0.004029
3,14593,2018-01-05,AAPL,APPLE INC,175.000000,0.011385,0.007034
4,14593,2018-01-08,AAPL,APPLE INC,174.350006,-0.003714,0.001662


In [24]:
# multiple conditions: AAPL, after 2020, with an above-average price
aapl_recent = df[
    (df["Ticker Symbol"] == "AAPL")
    & (df["Names Date"] > pd.to_datetime("2019-01-01"))
    & (df["Price or Bid/Ask Average"] < df["Price or Bid/Ask Average"].mean())
]
aapl_recent.head()
#aapl["Price or Bid/Ask Average"].mean()

,PERMNO,Names Date,Ticker Symbol,Company Name,Price or Bid/Ask Average,Returns,Return on the S&P 500 Index
1007,14593,2019-01-02,AAPL,APPLE INC,157.919998,0.001141,0.001269
1008,14593,2019-01-03,AAPL,APPLE INC,142.190002,-0.099607,-0.024757
1009,14593,2019-01-04,AAPL,APPLE INC,148.259995,0.042689,0.034336
1010,14593,2019-01-07,AAPL,APPLE INC,147.929993,-0.002226,0.007010
1011,14593,2019-01-08,AAPL,APPLE INC,150.750000,0.019063,0.009695


### Basic analytics

In [27]:
aapl["Returns"].describe()

count    756.000000
mean       0.001809
std        0.022105
min       -0.128647
25%       -0.007833
50%        0.001649
75%        0.012230
max        0.119808
Name: Returns, dtype: float64

A self-defined function (or a `lambda`) applied with `.map()` works the same way as it does
on a plain Python list, just column-by-column:

In [28]:
aapl["%returns"] = aapl["Returns"].map(lambda x: x * 100)
aapl[["Names Date", "Returns", "%returns"]].head()

,Names Date,Returns,%returns
0,2018-01-02,0.017905,1.790462
1,2018-01-03,-0.000174,-0.017415
2,2018-01-04,0.004645,0.464497
3,2018-01-05,0.011385,1.138532
4,2018-01-08,-0.003714,-0.371425


### GroupBy

In [32]:
groups = df.groupby("Ticker Symbol")
# groups is a DataFrameGroupBy object — not a DataFrame itself. 
# Nothing happens until you call something on it — e.g. groups["Returns"].describe() or groups.mean() — which is when pandas actually runs that operation separately on each ticker's sub-table and stitches the results back together.
groups["Returns"].describe()

,count,mean,std,min,25%,50%,75%,max
Ticker Symbol,,,,,,,,
AAPL,756.0,0.001809,0.022105,-0.128647,-0.007833,0.001649,0.012230,0.119808
AMZN,756.0,0.001574,0.020916,-0.079221,-0.007994,0.001702,0.011851,0.094452
GOOG,756.0,0.000870,0.019383,-0.111008,-0.007027,0.001416,0.011011,0.104485
NFLX,756.0,0.001733,0.026916,-0.111375,-0.012847,0.000876,0.017210,0.116087


In [38]:
# any function works, including your own
groups["Returns"].aggregate(["mean", "std", "count"])

,mean,std,count
Ticker Symbol,,,
AAPL,0.001809,0.022105,756
AMZN,0.001574,0.020916,756
GOOG,0.000870,0.019383,756
NFLX,0.001733,0.026916,756


<span style="color:#b45309">**Exercise 3: Which ticker has the most positive-return days?**</span>

Using `groups` from above, find which ticker has the largest number of days with a positive return (`Returns > 0`).

Hint: `(x > 0).sum()` inside `.aggregate(lambda x: ...)` counts positive values in each group; `.idxmax()` on the resulting Series gives you the group with the largest count.

Try it in the cell below, then check the answer notebook (`Session 2a - Exercise Answers.ipynb`).


In [ ]:
# your code here


Renaming columns matters right before a merge, too -- two tables almost never use exactly the same column name for what should be the same key, so `.rename()` is often the first step in lining them up:

In [ ]:
df.rename(columns={"Price or Bid/Ask Average": "Price"}).columns

## <span style="color:#1e3a8a">Merging</span>

Two different ways to combine tables: **concatenating** (stacking rows or columns together, no key involved) and **merging** (joining on a shared key). We'll look at both, starting with the simpler one.

### Concatenating (stacking) DataFrames

`pd.concat()` just stacks DataFrames on top of (or beside) each other -- it does **not** try to match rows by any key, it only cares that the columns line up. Useful when you have the *same kind* of data split across several tables (e.g. one file per year) and just want them combined.

In [ ]:
aapl_only = df[df["Ticker Symbol"] == "AAPL"]
goog_only = df[df["Ticker Symbol"] == "GOOG"]

# stacked back together -- same result as if we had never split them apart
stacked = pd.concat([aapl_only, goog_only])
stacked.shape, aapl_only.shape[0] + goog_only.shape[0]

Contrast that with merging, next: merging *aligns* rows by a shared key, rather than just stacking whatever you hand it.

### Merge types

**Merge types**, depending on how keys match between two tables:

- **1:1** &mdash; each key appears once in each table (e.g. one row per country in two tables).
- **1:m** (one-to-many) &mdash; a key appears once in the "one" table, multiple times in the "many"
  table (e.g. one company row, many days of prices).
- **m:m** (many-to-many) &mdash; rare, usually a sign your keys aren't well defined yet. Be careful:
  it silently produces a much bigger table than you expect (every match on the left pairs with every
  match on the right).

And **how** two tables combine when a key doesn't appear on both sides:

| `how=` | Keeps |
|---|---|
| `"inner"` | only keys present in *both* tables |
| `"left"` | all keys from the left table, `NaN` where the right has no match |
| `"right"` | all keys from the right table, `NaN` where the left has no match |
| `"outer"` | every key from either table, `NaN` wherever a side is missing |

Let's build a second, small table to merge with `df` &mdash; company info, indexed by ticker
(a classic **1:many** merge: one company row matches many price rows):

In [ ]:
company_info = pd.DataFrame({
    "Ticker Symbol": ["AAPL", "GOOG", "AMZN", "NFLX"],
    "Sector":        ["Technology", "Technology", "Consumer Discretionary", "Communication Services"],
    "Headquarters":  ["Cupertino, US", "Mountain View, US", "Seattle, US", "Los Gatos, US"],
    "Founded":       [1976, 1998, 1994, 1997],
})
company_info

In [ ]:
merged = df.merge(company_info, on="Ticker Symbol", how="left")
merged.head()

Always validate what kind of merge you *think* you're doing &mdash; `validate` raises an error
if your assumption is wrong, before a silently-wrong merge corrupts your analysis:

In [ ]:
merged_checked = df.merge(company_info, on="Ticker Symbol", how="left", validate="m:1")
print("Merge validated: many rows in df matched at most one row in company_info.")

<span style="color:#b45309">**Exercise 4: A one-to-one merge**</span>

Build a small DataFrame `latest_price` with one row per ticker (its most recent closing price -- `df.sort_values("Names Date").groupby("Ticker Symbol").tail(1)` gets you there). Merge it with `company_info` -- this time it should be a genuine **1:1** merge. Validate it as such.

Try it in the cell below, then check the answer notebook (`Session 2a - Exercise Answers.ipynb`).


In [ ]:
# your code here


## <span style="color:#1e3a8a">Regex essentials</span>

A **regular expression** is a search pattern for text. Python's built-in `re` module is the tool;
useful when a company name, ticker, or date shows up embedded in messier text than a clean column.


In [47]:
import re

text = "Q3 2024 revenue grew 12%, driven by AAPL and GOOG. Contact: ir@example.com"

Three core functions:
- `re.search(pattern, text)` &mdash; first match anywhere in the text
- `re.findall(pattern, text)` &mdash; *all* matches, as a list
- `re.sub(pattern, replacement, text)` &mdash; find and replace


In [50]:
re.search(r"AAPL", text)
# the start/end character positions where "AAPL" was found in text

<re.Match object; span=(36, 40), match='AAPL'>

In [52]:
# \d = a digit, + = one or more -> matches runs of digits
#  every run of one-or-more consecutive digits found in the text, as a list of strings
re.findall(r"\d+", text)

['3', '2024', '12']

In [54]:
# \w = alphanumeric character, common pattern for a simple email
# Extracts email addresses: one-or-more word/dot/dash characters, an @, then one-or-more word/dot/dash characters. 
re.findall(r"[\w.-]+@[\w.-]+", text)

['ir@example.com']

In [55]:
# character classes: match either of two tickers
# Finds every occurrence of either "AAPL" or "GOOG" (the | means "or").
re.findall(r"AAPL|GOOG", text)

['AAPL', 'GOOG']

In [56]:
# anonymize: replace every all-caps ticker-looking token
# Replaces every standalone all-caps word of 2–5 letters (like AAPL, GOOG) with [TICKER]. \b = word boundary, [A-Z]{2,5} = 2 to 5 uppercase letters.
re.sub(r"\b[A-Z]{2,5}\b", "[TICKER]", text)

'Q3 2024 revenue grew 12%, driven by [TICKER] and [TICKER]. Contact: ir@example.com'

<span style="color:#b45309">**Exercise 5: Extract every 4-digit year from a headline**</span>

From `"London Olympics 2012 was followed by Rio 2016 and Tokyo 2021."`, extract every 4-digit year using `re.findall`.

Try it in the cell below, then check the answer notebook (`Session 2a - Exercise Answers.ipynb`).


In [ ]:
# your code here


## <span style="color:#1e3a8a">Fuzzy matching </span>

Merging works perfectly when both tables share a clean key. In the real world, they often don't:
the *same* company can appear as `"W W INTERNATIONAL INC"`, `"WW International, Inc."`, and
`"WW International"` across two datasets. **CS1** is built entirely around exactly this problem.

**Fuzzy matching** scores how similar two strings are, so you can match on "close enough" rather
than "identical." The `thefuzz` library implements this via the **Levenshtein distance** (roughly:
how many single-character edits turn one string into the other).


In [57]:
#!pip install thefuzz python-Levenshtein
from thefuzz import fuzz, process

In [61]:
a = "WW International Inc"
b = "W W INTERNATIONAL, INC."

print("ratio:          ", fuzz.ratio(a, b))                # plain character-level similarity
print("token_sort_ratio:", fuzz.token_sort_ratio(a, b))     # ignores word order / fuzz.token_sort_ratio compares two strings after splitting each into words, sorting those words alphabetically, then rejoining and comparing — so word order doesn't matter. E.g. "New York Mets" and "Mets New York" score low on plain fuzz.ratio (different order) but high here, since after sorting both become the same word sequence.
print("token_set_ratio: ", fuzz.token_set_ratio(a, b))      # ignores word order AND duplicate/extra words


ratio:           28
token_sort_ratio: 98
token_set_ratio:  97


Every thefuzz scorer (ratio, partial_ratio, token_sort_ratio, token_set_ratio, WRatio) returns an integer from 0 to 100, where 100 = identical (after whatever normalization that scorer does) and 0 = nothing alike.

`token_sort_ratio` and `token_set_ratio` matter because company names get reordered or
padded ("Inc", "Ltd", "Corp") in ways plain `ratio` is too literal to see past. `WRatio` (weighted
ratio) combines several of these heuristics and is a reasonable default when you're not sure which
to pick.

In [60]:
print(fuzz.WRatio(a, b))

95


**`process.extractOne`** finds the best match for one string out of a whole list of candidates
&mdash; exactly what you need to fuzzy-merge one dataset's names against another's:

In [62]:
candidates = ["Apple Inc", "Alphabet Inc", "Amazon.com Inc", "Netflix Inc", "WW International Inc"]

process.extractOne("W.W. Internatinal, Inc", candidates, scorer=fuzz.token_sort_ratio)

('WW International Inc', 95)

<span style="color:#b45309">**Exercise 6: Fuzzy-merge two small tables**</span>

Two DataFrames below share the same real-world entities under slightly different spellings. For each `Key` in `df1`, find its best-matching `Key` in `df2` (use `process.extractOne`), store it in a new column `MergeKey`, then merge `df1` and `df2` on that key.

How many of the 4 rows in `df1` found a sensible match in `df2`?

Try it in the cell below, then check the answer notebook (`Session 2a - Exercise Answers.ipynb`).


In [ ]:
df1 = pd.DataFrame({"Key": ["Apple", "Banana", "Orange", "Strawberry"], "val1": [10, 28, 13, 18]})
df2 = pd.DataFrame({"Key": ["Aple", "Mango", "Orag", "Straw", "Bannanna", "Berry"], "val2": [.1, .5, .5, .1, .3, .2]})


In [ ]:
# your code here


### Looking ahead to CS1

CS1 asks you to link earnings-call records to stock-return records using exactly this toolkit &mdash;
except the "obvious" match isn't always the *right* one (a high fuzzy score doesn't mean the merge is
correct: company name changes, M&amp;A, and near-duplicate names can all fool a similarity score).
Treat a fuzzy match as a **candidate** to validate, not a guaranteed answer &mdash; the case brief asks
you directly what could go wrong and how you'd check.
